# 00 · Business Understanding — Rossmann Store Sales

This notebook frames the machine learning problem before any code is written.  
Following Aurélien Géron's *Hands-On ML* Chapter 2 checklist:

> *"Before you start writing code, you should step back and think about what the overall objective is."*

---

## 1. Executive context

Rossmann operates **over 3,000 drugstores** across 7 European countries.  
The CFO needs to know: **how much will each store sell in the next 6 weeks?**

This forecast supports:
- Capital allocation and store renovation scheduling
- Cash-flow planning
- Staffing and inventory decisions

---

## 2. Problem framing (ML project checklist)

| Question | Answer |
|---|---|
| **Task type** | Supervised regression (time-series) |
| **Target variable** | `Sales` — daily revenue per store |
| **Prediction horizon** | 6 weeks ahead |
| **Granularity** | One prediction per store-day |
| **Performance measure** | MAPE and RMSPE (relative errors, scale-invariant across stores) |
| **Baseline** | Average historical sales per store |
| **Output consumers** | CFO dashboard · Telegram bot · REST API |

---

## 3. Data source

Kaggle competition: [Rossmann Store Sales](https://www.kaggle.com/competitions/rossmann-store-sales)

| File | Rows | Description |
|---|---|---|
| `train.csv` | 1,017,209 | Historical daily sales per store (2013-01-01 → 2015-07-31) |
| `test.csv` | 41,088 | Stores + dates for which we must predict (no target) |
| `store.csv` | 1,115 | Store metadata (type, assortment, competition, promotions) |
| `sample_submission.csv` | 41,088 | Expected submission format |

---

## 4. Success criteria

1. **Temporal validation** — the hold-out set must be the *last N weeks*, never a random split, to prevent data leakage.
2. **MAPE < 12%** on the validation window (comparable to top Kaggle solutions).
3. **Business translation** — express the model error as a financial range: minimum / expected / maximum scenario.
4. **Operational delivery** — a recruiter or CFO can query store #42 via Telegram and get a forecast in seconds.

---

## 5. Assumptions

- Closed stores (`Open == 0`) generate zero revenue and are excluded from both training and evaluation.
- The `Sales` target is log-transformed (`log1p`) during training to reduce skewness and stabilise gradient updates — a common pre-processing choice for right-skewed regression targets.
- Promotions, holidays, store type, assortment, competition proximity and time-based cyclical features are the primary drivers of sales variance.

---

## 6. Project flow

```
00_business_understanding  →  frame the problem
01_data_understanding      →  load, profile, create time-split
02_exploratory_analysis    →  visualise, correlate, test hypotheses
03_feature_engineering     →  custom sklearn transformer, pipeline
04_modeling_and_business   →  train, evaluate, translate to cash
05_deployment              →  FastAPI + Telegram bot demo
```

In [ ]:
from pathlib import Path

# ── repository root (works from any working directory) ───────────────────────
ROOT = Path.cwd()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

print('Project root:', ROOT)

import sys
sys.path.insert(0, str(ROOT / 'src'))

from rossmann_store_sales.config import load_config

cfg = load_config(ROOT / 'configs' / 'project.toml')
print('\nProject config:')
for section, values in cfg.items():
    if section != '_project_root':
        print(f'  [{section}]', values)